# Fabric Liquid Clustering Benchmark (Sanitized)

This companion notebook compares standard Delta optimization with Liquid Clustering on a synthetic clinical-observation dataset. It is a platform benchmark only and does not train or evaluate a clinical prediction model.

Saved outputs, workspace identifiers, table names, session identifiers, and patient identifiers have been removed. Replace every placeholder with approved values in your Fabric workspace before running.

In [ ]:
import statistics
import time

# Configure approved Fabric objects. Do not commit workspace-specific values.
catalog = "<catalog_name>"
schema = "<schema_name>"
source_table = f"{catalog}.{schema}.observations"
control_table = f"{catalog}.{schema}.observations_control"
liquid_table = f"{catalog}.{schema}.observations_liquid"

if catalog.startswith("<") or schema.startswith("<"):
    raise ValueError("Set approved catalog and schema values before running.")

## Create comparable Delta tables

This cell intentionally stops rather than overwriting an existing table. Review storage cost and access controls before running it.

In [ ]:
for table_name in [control_table, liquid_table]:
    if spark.catalog.tableExists(table_name):
        raise RuntimeError(f"{table_name} already exists. Review before proceeding.")

spark.sql(f"CREATE TABLE {control_table} USING DELTA AS SELECT * FROM {source_table}")
spark.sql(f"CREATE TABLE {liquid_table} CLUSTER BY (code, date, patient) AS SELECT * FROM {source_table}")
spark.sql(f"OPTIMIZE {control_table}")
spark.sql(f"OPTIMIZE {liquid_table}")

## Verify physical layout

Record only aggregate metadata. Do not export table paths, workspace identifiers, patient records, or session information.

In [ ]:
for table_name in [control_table, liquid_table]:
    print(f"TABLE: {table_name}")
    spark.sql(f"DESCRIBE DETAIL {table_name}").select(
        "format", "numFiles", "sizeInBytes", "clusteringColumns"
    ).show(truncate=False)

## Repeated query benchmark

The queries below use only aggregate results. Run each configuration five times, alternate execution order, and compare medians.

In [ ]:
query_templates = {
    "code_filter": """
        SELECT code, COUNT(*) AS total_observations
        FROM {table}
        WHERE code = '<observation_code>'
        GROUP BY code
    """,
    "code_date_filter": """
        SELECT code, COUNT(*) AS total_observations
        FROM {table}
        WHERE code = '<observation_code>'
          AND date >= '<start_date>'
          AND date < '<end_date>'
        GROUP BY code
    """,
}

results = []
for query_name, query_template in query_templates.items():
    timings = {"control": [], "liquid": []}
    outputs = {}

    for label, table_name in [("control", control_table), ("liquid", liquid_table)]:
        spark.sql(query_template.format(table=table_name)).collect()

    for run in range(5):
        order = [("control", control_table), ("liquid", liquid_table)]
        if run % 2:
            order.reverse()
        for label, table_name in order:
            spark.catalog.clearCache()
            started = time.perf_counter()
            rows = spark.sql(query_template.format(table=table_name)).collect()
            timings[label].append(time.perf_counter() - started)
            outputs[label] = sorted([tuple(row) for row in rows], key=str)

    if outputs["control"] != outputs["liquid"]:
        raise ValueError(f"Result mismatch for {query_name}")

    control_median = statistics.median(timings["control"])
    liquid_median = statistics.median(timings["liquid"])
    results.append({
        "query": query_name,
        "control_median_seconds": control_median,
        "liquid_median_seconds": liquid_median,
        "relative_change_percent": (control_median - liquid_median) / control_median * 100,
    })

results